# **C05 - Valuación con datos de mercado y separación train-test**
## **Simulación de Procesos Financieros**

---

**Autor:** Francisco Uriel Ledezma Chávez  
**Carrera:** Ingeniería Financiera  
**Institución:** ITESO — Universidad Jesuita de Guadalajara  
**Semestre:** 5° — Simulación de Procesos Financieros  
**Fecha de realización:** 23 de agosto de 2026

---

En este notebook se estiman los parámetros de un proceso de precios a partir de datos reales de mercado, se separa la muestra en un tramo de entrenamiento y otro de prueba para evitar que la información del periodo evaluado contamine la estimación, y se simulan 10,000 trayectorias del precio del activo con el propósito de delimitar el rango de valores razonables al cierre del horizonte.

## Índice de contenidos

1. Sección 1 — Ingesta de precios
2. Sección 2 — Rendimientos diarios
3. Sección 3 — Separación train-test
4. Sección 4 — Estimación de parámetros
5. Sección 5 — Simulación del precio al horizonte
6. Sección 6 — Rango razonable de precios
7. Sección 7 — Validación contra el precio observado
8. Sección 8 — Conclusiones

## Importación de bibliotecas

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

## 1. Ingesta de precios

Se descargan los precios de cierre de Coca-Cola entre el primero de enero y el 31 de agosto de 2026, periodo suficientemente amplio para separar después un tramo de estimación y otro de evaluación sin que ninguno quede demasiado corto.

In [17]:
Prices = yf.download('KO', start='2026-01-01', end='2026-08-31')['Close']
Prices.dropna(inplace=True)
Prices

[*********************100%***********************]  1 of 1 completed


Ticker,KO
Date,
2026-01-02,68.207611
2026-01-05,67.043182
2026-01-06,66.944496
2026-01-07,66.648468
2026-01-08,68.454315
...,...
2026-08-24,91.989998
2026-08-25,91.639999
2026-08-26,90.080002


La descarga entrega 165 sesiones de cotización, número que corresponde a los días hábiles del periodo y que conviene tener presente porque de él dependen los tamaños de los dos subconjuntos que se construyen más adelante.

## 2. Rendimientos diarios

El rendimiento se calcula como la variación relativa entre cierres consecutivos, de modo que la serie resultante pierde la primera observación por no tener referencia previa.

$$R_t = \frac{P_t}{P_{t-1}} - 1$$

In [18]:
Rendimientos = (Prices / Prices.shift(1) - 1).dropna()
Rendimientos

Ticker,KO
Date,
2026-01-05,-0.017072
2026-01-06,-0.001472
2026-01-07,-0.004422
2026-01-08,0.027095
2026-01-09,0.016434
...,...
2026-08-24,0.009769
2026-08-25,-0.003805
2026-08-26,-0.017023


La serie de rendimientos queda con 164 observaciones frente a las 165 sesiones de precios, y esa unidad de diferencia no es un detalle menor cuando se definen las ventanas, ya que cualquier conteo posterior debe hacerse sobre los rendimientos y no sobre los precios.

## 3. Separación train-test

La muestra se corta en dos tramos disjuntos, el primero anterior al primero de mayo para estimar los parámetros y el segundo entre esa fecha y el primero de julio para evaluar, criterio que respeta el orden temporal y evita que información posterior al corte influya sobre la estimación.

In [19]:
Train = Rendimientos[Rendimientos.index < "2026-05-01"]
Test = Rendimientos[(Rendimientos.index >= "2026-05-01") & (Rendimientos.index < "2026-07-01")]

In [20]:
len(Train)

81

In [21]:
len(Test)

41

El tramo de entrenamiento reúne 81 sesiones y el de prueba 41, proporción cercana a dos tercios contra un tercio que resulta razonable para una serie de este tamaño, y el segundo número tiene además un papel directo en la simulación, ya que 41 es exactamente el número de días hábiles que separan el corte del horizonte a evaluar.

## 4. Estimación de parámetros

La media y la desviación se calculan únicamente sobre el tramo de entrenamiento, y ambas quedan expresadas en frecuencia diaria porque los rendimientos lo están, condición que obliga a medir después el horizonte en días y no en años.

In [22]:
Mu = Train.mean()
Sigma = Train.std()

print(Mu, Sigma)

Ticker
KO    0.001763
dtype: float64 Ticker
KO    0.011509
dtype: float64


La media diaria estimada es 0.001763 y la desviación 0.011509, es decir, el rendimiento esperado de una sesión equivale a 15% de su propia desviación, relación que anticipa un comportamiento dominado por el ruido, ya que en el corto plazo la dispersión pesa mucho más que la tendencia.

Llevadas a escala anual con 252 sesiones, esas cifras equivalen a una deriva cercana a 44% y una volatilidad aproximada de 18%, valores que conviene interpretar con cautela, pues la deriva anualizada resulta muy alta para una empresa de consumo estable y refleja que el tramo de entrenamiento cayó dentro de una fase alcista particular.

Los parámetros de la simulación se fijan con lo estimado, tomando como precio inicial el último cierre del tramo de entrenamiento y como horizonte el número de sesiones del tramo de prueba, de manera que la simulación proyecta exactamente el periodo que se desea evaluar.

In [23]:
S_0 = Prices.loc[:'2026-04-30'].iloc[-1]
r = Mu
sigma = Sigma
T = len(Test)
K = Prices.loc['2026-07-01']


In [24]:
S_0

Ticker
KO    78.254768
Name: 2026-04-30 00:00:00, dtype: float64

El precio de partida es 78.254768, correspondiente al cierre del 30 de abril, y el horizonte queda en 41 sesiones, con lo cual la proyección termina el primero de julio.

## 5. Simulación del precio al horizonte

Cada trayectoria se genera con el modelo de movimiento browniano geométrico usando los parámetros diarios y el horizonte en días, y el experimento se repite 10,000 veces para construir la distribución del precio al cierre del periodo de prueba.

$$S_T = S_0\exp\left[\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}Z\right]$$

In [25]:
np.random.seed(23)
resultados = []

for i in range(10000):
    S_t = S_0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * np.random.normal(0, 1))
    resultados.append(S_t)

len(resultados)


10000

## 6. Rango razonable de precios

**Pregunta:** ¿cuál es el peor precio razonable para considerar?

La respuesta exige distinguir entre dos ideas que suelen confundirse, ya que el peor caso absoluto de una simulación carece de utilidad porque siempre puede empeorar al agregar trayectorias, mientras el peor caso razonable corresponde a un percentil bajo de la distribución, es decir, a un nivel que sólo se rebasa con una probabilidad pequeña y prefijada.

In [26]:
resultados = np.array(resultados)

Como referencia adicional se calcula el precio que resultaría de proyectar la deriva estimada de manera lineal sobre el horizonte completo, valor que sirve de contraste porque ignora por completo la incertidumbre y por ello debe quedar cerca del centro de la distribución simulada.

In [27]:
M = S_0 + S_0 * (Train.mean()) * T
M


Ticker
KO    83.91127
dtype: float64

In [28]:
P_menor = np.percentile(resultados, 5)
P_menor

np.float64(74.26227850399476)

In [29]:
P_mayor = np.percentile(resultados, 95)
P_mayor


np.float64(94.63810485977513)

El percentil 5 delimita el peor precio razonable bajo el criterio adoptado, ya que sólo 5% de las trayectorias termina por debajo de ese nivel, y el percentil 95 cumple la función simétrica del lado favorable, de modo que el intervalo entre ambos concentra 90% de los desenlaces simulados y constituye el rango que un tomador de decisiones debería considerar plausible.

Los valores obtenidos son 74.262279 y 94.638105, es decir, el peor precio razonable queda 5.10% por debajo del punto de partida mientras el mejor lo supera en 20.94%, y esa asimetría respecto del precio inicial no proviene del método sino de la deriva estimada, ya que 41 sesiones a una media diaria de 0.001763 acumulan un crecimiento esperado de 7.23% que desplaza la distribución completa hacia arriba antes de que la incertidumbre la abra en ambos sentidos.

La proyección lineal entrega el contraste anunciado, pues sus 83.911270 quedan a 0.018672 de la mediana teórica del proceso, $S_0e^{(\mu-\sigma^2/2)T} = 83.892598$, y sin embargo se sitúan a 11.50% del extremo inferior del intervalo y a 12.78% del superior, de modo que ignorar la incertidumbre no desplaza el nivel central pero elimina por completo la información sobre el rango, que es justamente lo que se necesita para dimensionar el riesgo.

## 7. Validación contra el precio observado

La proyección sólo adquiere sentido si se contrasta con lo que efectivamente ocurrió, y para eso sirve el precio del primero de julio que se había guardado como referencia sin utilizarse hasta ahora, ya que ubicarlo dentro de la distribución simulada permite juzgar si el modelo calibrado con el tramo de entrenamiento describe adecuadamente el comportamiento posterior del activo.

In [30]:
precio_observado = float(K.iloc[0])
percentil_observado = (resultados < precio_observado).mean() * 100

print(f"Precio observado el 1 de julio: {precio_observado:.6f}")
print(f"Dentro del intervalo del 90%: {P_menor <= precio_observado <= P_mayor}")
print(f"Percentil que ocupa en la distribución simulada: {percentil_observado:.2f}")

Precio observado el 1 de julio: 81.290001
Dentro del intervalo del 90%: True
Percentil que ocupa en la distribución simulada: 34.40


El precio observado fue 81.290001 y cae dentro del intervalo del 90%, ocupando el percentil 34.40 de la distribución simulada, de modo que el modelo no queda desmentido por la evidencia, ya que un desenlace en el tercio inferior del rango es perfectamente compatible con lo que la simulación consideraba plausible.

El percentil empírico admite además una comprobación analítica, pues el rendimiento logarítmico observado se separa 0.4276 desviaciones de la deriva esperada y la normal acumulada en ese punto vale 33.45%, cifra que reproduce el 34.40 obtenido por conteo sobre las 10,000 trayectorias y confirma que la simulación es consistente con el modelo que dice implementar.

El resultado deja ver con claridad dónde falló la calibración, ya que el activo avanzó 3.8787% en las 41 sesiones mientras la deriva estimada anticipaba 7.2283%, es decir, el rendimiento realizado fue poco más de la mitad del proyectado y el precio terminó 3.10% por debajo de la mediana, lo cual confirma que la media muestral del tramo de entrenamiento sobreestimó la tendencia, en cambio la dispersión sí resultó adecuada, dado que el desenlace quedó holgadamente dentro de la banda que la volatilidad estimada había delimitado.

## 8. Conclusiones

La separación train-test cumple aquí una función concreta y no meramente formal, ya que los parámetros se estimaron con 81 sesiones anteriores al corte y la proyección se construyó sobre las 41 sesiones siguientes, con lo cual ningún dato del periodo evaluado participó en la estimación y el ejercicio conserva el carácter de una predicción genuina.

El resultado más relevante es la asimetría entre los dos parámetros estimados, pues una media diaria de 0.001763 frente a una desviación de 0.011509 implica que la incertidumbre acumulada domina ampliamente a la tendencia en un horizonte de 41 sesiones, y de ahí que el intervalo del 90% abarque de 74.262279 a 94.638105, una amplitud de 20.375826 pesos equivalente a 26.04% del precio de partida, aun cuando la deriva estimada sea positiva.

Ese predominio admite una medición directa, ya que sobre el horizonte completo la deriva acumulada es 7.23% mientras la volatilidad acumulada, $\sigma\sqrt{T}$, alcanza 7.37%, de manera que la incertidumbre de una sola desviación estándar ya supera a todo el crecimiento esperado del periodo.

La validación contra el dato observado permite separar dos juicios que suelen mezclarse, pues el precio del primero de julio cayó dentro del intervalo del 90% en el percentil 34.40 y por ello la banda resultó correcta, mientras el rendimiento realizado de 3.8787% se quedó en poco más de la mitad del 7.2283% proyectado, de manera que el modelo acertó en la magnitud del riesgo y falló en la dirección esperada, lo cual es exactamente el patrón que anticipaba la comparación entre deriva y volatilidad acumuladas.

Conviene reconocer los límites que subsisten, ya que la deriva se estimó con la media de rendimientos simples cuando el modelo lognormal requiere la de rendimientos logarítmicos, diferencia pequeña en frecuencia diaria pero conceptualmente relevante, y sobre todo porque una sola observación no valida ni refuta un modelo, de modo que la continuación natural consiste en repetir el procedimiento sobre ventanas sucesivas y verificar si la proporción de desenlaces que caen dentro del intervalo se acerca al 90% comprometido.